# 解答① SFT + LoRA

> **講師用**: 演習 `ex_01_sft.ipynb` の完全解答です。参加者には学習中は非公開にしてください。

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# 解答: BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
) if device == 'cuda' else None

MODEL_NAME = 'meta-llama/Meta-Llama-3-8B'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto' if device == 'cuda' else None,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 解答: LoraConfig
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 解答: format_instruction
def format_instruction(sample: dict) -> str:
    instruction = sample.get('instruction', '')
    context     = sample.get('context', '')
    response    = sample.get('response', '')
    if context:
        return (
            f'### 指示:\n{instruction}\n\n'
            f'### 文脈:\n{context}\n\n'
            f'### 回答:\n{response}'
        )
    return f'### 指示:\n{instruction}\n\n### 回答:\n{response}'

# テスト
test = {'instruction': 'Pythonとは？', 'context': '', 'response': 'プログラミング言語です。'}
print(format_instruction(test))

In [ ]:
# 解答: SFTConfig
dataset = load_dataset('kunishou/databricks-dolly-15k-ja', split='train')
dataset = dataset.map(
    lambda x: {'text': format_instruction(x)},
    remove_columns=dataset.column_names,
)

training_args = SFTConfig(
    output_dir='./outputs/ex01_sft',
    max_steps=20,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=device == 'cuda',
    logging_steps=5,
    report_to='none',
    max_seq_length=512,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
trainer.train()
print('学習完了！')

In [ ]:
# 実験課題の解答: rank による学習パラメータ数の比較
from peft import LoraConfig, get_peft_model

for rank in [8, 16, 32, 64]:
    lc = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank, lora_alpha=rank*2,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        bias='none',
    )
    # 学習可能パラメータ数のみ計算（実際にモデルを作成する場合はコメントを外す）
    # m = get_peft_model(base_model, lc)
    # m.print_trainable_parameters()
    d = 4096  # LLaMA 3 8B の hidden_dim
    n_layers = 32
    modules = 4  # q, k, v, o
    params = n_layers * modules * 2 * d * rank
    print(f'rank={rank:3d}: 推定学習パラメータ数 ≈ {params:,}')